# Proyecto 2P - Clasificacion del Riesgo de Inundacion por Parroquia
**Provincia: Esmeraldas, Ecuador**

> **Dataset enriquecido**: 6 variables climaticas/geograficas + 13 variables del censo INEC 2022
> (poblacion, hogar, vivienda, emigrantes) + datos MAATE (concesiones de agua).

**Variables predictoras originales:**
- Precipitacion acumulada promedio anual (mm)
- Altitud media (m)
- Pendiente del terreno (%)
- Distancia media a cuerpos de agua (km)
- Densidad poblacional (hab/km2)
- Porcentaje de area urbanizada

**Variables INEC incorporadas:**
- Edad media, % menores 15, % mayores 65, % hombres
- % agua publica, % alcantarillado, % electricidad, hacinamiento
- % pared buena, % techo bueno, % piso bueno, personas por dormitorio
- Total emigrantes

**Variable objetivo:** Riesgo de inundacion (Bajo / Medio / Alto)

**Modelos evaluados:** Logistic Regression, Decision Tree, Random Forest, SVM, Voting Classifier


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, warnings, os, joblib
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV, learning_curve
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')


## 1. Carga del Dataset Enriquecido


In [ ]:
DATA_DIR = r'C:\Users\USER\Desktop\Proyecto Aprendizaje automatico'

# Dataset enriquecido con INEC + MAATE
df = pd.read_csv(os.path.join(DATA_DIR, 'dataset_riesgo_inundacion_enriquecido.csv'), encoding='utf-8-sig')
print(f'Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas')
print(f'Parroquias: {df[\"PARROQUIA\"].nunique()}, Cantones: {df[\"CANTON\"].nunique()}')

# Features originales + INEC
FEATURES_ORIG = ['PRECIPITACION_ANUAL_MM','ALTITUD_MEDIA_M','PENDIENTE_PCT','DISTANCIA_AGUA_KM','DENSIDAD_POBLACIONAL','AREA_URBANIZADA_PCT']
FEATURES_INEC = ['POB_EDAD_MEDIA','POB_PCT_MENORES_15','POB_PCT_MAYORES_65','POB_PCT_HOMBRES','HOG_PCT_AGUA','HOG_PCT_ALCANTARILLADO','HOG_PCT_ELECTRICIDAD','HOG_HACINAMIENTO','VIV_PCT_PARED_BUENA','VIV_PCT_TECHO_BUENO','VIV_PCT_PISO_BUENO','VIV_PERS_POR_DORM','EMI_TOTAL']
ALL_FEATURES = FEATURES_ORIG + FEATURES_INEC
print(f'Features originales: {len(FEATURES_ORIG)}')
print(f'Features INEC: {len(FEATURES_INEC)}')


## 2. Analisis Exploratorio de Datos (EDA)


In [ ]:
print('--- Distribucion variable objetivo ---')
print(df['RIESGO_INUNDACION'].value_counts())
print(df['RIESGO_INUNDACION'].value_counts(normalize=True) * 100)

riesgo_colors = {'Bajo': '#2ecc71', 'Medio': '#f39c12', 'Alto': '#e74c3c'}
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    feat = FEATURES_ORIG[i]
    for riesgo, color in riesgo_colors.items():
        subset = df[df['RIESGO_INUNDACION'] == riesgo][feat]
        ax.hist(subset, alpha=0.6, label=riesgo, bins=10, color=color)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=12)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Correlacion con features INEC
fig, ax = plt.subplots(figsize=(12, 8))
corr = df[ALL_FEATURES + ['RIESGO_NUM']].corr()
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f', ax=ax, linewidths=0.5)
ax.set_title('Matriz de Correlacion (19 features)', fontsize=14)
plt.tight_layout()
plt.show()


## 3. Preprocesamiento


In [ ]:
# Llenar NaN en features INEC
for c in FEATURES_INEC:
    if c in df.columns:
        df[c] = df[c].fillna(0)

X = df[ALL_FEATURES].values
y = df['RIESGO_INUNDACION'].values

le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f'Clases: {list(le.classes_)} -> {list(le.transform(le.classes_))}')

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded)
print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')


## 4. Entrenamiento y Evaluacion de Modelos


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1),
    'SVM (RBF)': SVC(C=10, gamma='scale', kernel='rbf', probability=True, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for name, model in models.items():
    pipeline = Pipeline([('scaler', RobustScaler()), ('clf', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1_weighted')
    results.append({
        'Modelo': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred, average='weighted'),
        'CV F1 (mean)': cv_scores.mean(),
        'CV F1 (std)': cv_scores.std(),
    })
    print(f'\\n{"="*50}')
    print(name)
    print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(f'Test F1-Score: {f1_score(y_test, y_pred, average="weighted"):.4f}')
    print(f'CV F1: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
    print(f'\\n{classification_report(y_test, y_pred, target_names=le.classes_)}')


## 5. Voting Classifier (Ensemble)


In [ ]:
voting_soft = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
        ('rf', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)),
        ('svm', SVC(C=10, gamma='scale', probability=True, random_state=42)),
    ], voting='soft')

for vname, vclf in [('Voting Soft', voting_soft)]:
    pipeline_v = Pipeline([('scaler', RobustScaler()), ('clf', vclf)])
    pipeline_v.fit(X_train, y_train)
    y_pred_v = pipeline_v.predict(X_test)
    cv_scores_v = cross_val_score(pipeline_v, X_train, y_train, cv=cv, scoring='f1_weighted')
    results.append({
        'Modelo': vname,
        'Accuracy': accuracy_score(y_test, y_pred_v),
        'Precision': precision_score(y_test, y_pred_v, average='weighted'),
        'Recall': recall_score(y_test, y_pred_v, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred_v, average='weighted'),
        'CV F1 (mean)': cv_scores_v.mean(),
        'CV F1 (std)': cv_scores_v.std(),
    })
    print(f'{vname}: Acc={accuracy_score(y_test, y_pred_v):.4f}, F1={f1_score(y_test, y_pred_v, average="weighted"):.4f}')


## 6. Tabla Comparativa


In [ ]:
df_results = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
display(df_results)


## 7. Optimizacion de Hiperparametros (Random Forest)


In [ ]:
param_grid = {
    'clf__n_estimators': [100, 200, 300],
    'clf__max_depth': [5, 8, 12, None],
    'clf__min_samples_split': [2, 4, 6],
    'clf__min_samples_leaf': [1, 2, 4],
}

pipeline_rf = Pipeline([('scaler', RobustScaler()), ('clf', RandomForestClassifier(random_state=42, n_jobs=-1))])
grid_search = GridSearchCV(pipeline_rf, param_grid, cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(f'Mejores params: {grid_search.best_params_}')
print(f'Mejor CV F1: {grid_search.best_score_:.4f}')

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
best_acc = accuracy_score(y_test, y_pred_best)
best_f1 = f1_score(y_test, y_pred_best, average='weighted')
print(f'Test Accuracy (opt): {best_acc:.4f}')
print(f'Test F1-Score (opt): {best_f1:.4f}')


## 8. Matriz de Confusion


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title(f'Matriz de Confusion - RF Optimizado\\nF1: {best_f1:.4f}, Acc: {best_acc:.4f}', fontsize=13)
plt.tight_layout(); plt.show()


## 9. Importancia de Caracteristicas


In [ ]:
rf_best = best_model.named_steps['clf']
importances = rf_best.feature_importances_
indices = np.argsort(importances)[::-1]

print('Top-10 variables mas importantes:')
for i in range(10):
    print(f'  {i+1}. {ALL_FEATURES[indices[i]]}: {importances[indices[i]]:.4f}')

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = plt.cm.Blues(np.linspace(0.4, 0.9, 10))
ax.barh(range(10), importances[indices[:10]], color=colors_bar)
ax.set_yticks(range(10))
ax.set_yticklabels([ALL_FEATURES[i] for i in indices[:10]])
ax.set_xlabel('Importancia', fontsize=12)
ax.set_title('Importancia de Caracteristicas (19 features)', fontsize=14)
ax.invert_yaxis(); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout(); plt.show()


## 10. Curva de Aprendizaje


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
train_sizes, train_scores, test_scores = learning_curve(best_model, X_train, y_train, cv=5, scoring='f1_weighted', n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 10))
train_mean = np.mean(train_scores, axis=1); train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1); test_std = np.std(test_scores, axis=1)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
ax.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.2, color='orange')
ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Entrenamiento')
ax.plot(train_sizes, test_mean, 'o-', color='orange', label='Validacion')
ax.set_xlabel('Tamano del conjunto de entrenamiento', fontsize=12)
ax.set_ylabel('F1-Score (weighted)', fontsize=12)
ax.set_title('Curva de Aprendizaje - RF (19 features)', fontsize=14)
ax.legend(loc='best'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 11. Exportacion del Modelo


In [ ]:
joblib.dump(best_model, os.path.join(DATA_DIR, 'modelo_riesgo_inundacion_rf_optimizado.pkl'))
joblib.dump(le, os.path.join(DATA_DIR, 'label_encoder.pkl'))
print('Modelo RF optimizado exportado (dataset enriquecido)')


## 12. Conclusiones


In [ ]:
best_row = df_results.iloc[0]
print(f'Mejor modelo: {best_row["Modelo"]}')
print(f'F1-Score: {best_row["F1-Score"]:.4f}')
print(f'Accuracy: {best_row["Accuracy"]:.4f}')
print()
print('='*60)
print('CONCLUSION')
print('='*60)
print('Se desarrollo un modelo de clasificacion de riesgo de inundacion para las parroquias')
print('de la provincia de Esmeraldas, Ecuador, utilizando 19 variables predictoras:')
print('6 climaticas/geograficas originales y 13 del censo INEC 2022 (poblacion, hogar, vivienda, emigrantes).')
print()
best_acc = accuracy_score(y_test, y_pred_best)
best_f1 = f1_score(y_test, y_pred_best, average='weighted')
best_cv = grid_search.best_score_
print(f'El modelo Random Forest optimizado alcanzo:')
print(f'  - Accuracy en test: {best_acc*100:.2f}%')
print(f'  - F1-Score en test: {best_f1*100:.2f}%')
print(f'  - CV F1-Score: {best_cv*100:.2f}%')
print()
print('Las variables mas influyentes fueron:')
for i in range(3):
    print(f'  {i+1}. {ALL_FEATURES[indices[i]]}: {importances[indices[i]]:.2%}')
print()
print('El modelo exportado se integro en una aplicacion Flask con mapa Leaflet, disponible en')
print('PythonAnywhere/Render/Railway para visualizacion interactiva del riesgo de inundacion por parroquia.')
